In [0]:
# ============================================================
# DATA VALIDATION & QUALITY CHECKS - UPDATED FOR CI/CD
# ============================================================

import os
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic paths based on environment
SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD   = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

print("✅ Config ready!")

# 

✅ Config ready!


In [0]:
df_txn   = spark.read.format("delta").load(f"{SILVER}/transactions")
df_deliq = spark.read.format("delta").load(f"{SILVER}/repayment_delinquency")
df_acc   = spark.read.format("delta").load(f"{SILVER}/account_master")
fact_txn = spark.read.format("delta").load(f"{GOLD}/fact_transactions")

print("✅ All tables loaded!")

✅ All tables loaded!


In [0]:
print("=" * 60)
print("  BUSINESS RULE VALIDATION REPORT")
print("=" * 60)

results = []

# ── Rule 1: amount >= 0 ──
r1 = df_txn.filter(F.col("amount") < 0).count()
results.append(("Rule 1", "amount >= 0",             r1))

# ── Rule 2: outstanding_balance >= 0 ──
r2 = df_txn.filter(F.col("outstanding_balance") < 0).count()
results.append(("Rule 2", "outstanding_balance >= 0", r2))

# ── Rule 3: credit_limit >= outstanding_balance ──
r3 = fact_txn.filter(F.col("outstanding_balance") > F.col("credit_limit")).count()
results.append(("Rule 3", "credit_limit >= outstanding_balance", r3))

# ── Rule 4: days_past_due >= 0 ──
r4 = df_deliq.filter(F.col("days_past_due") < 0).count()
results.append(("Rule 4", "days_past_due >= 0",       r4))

# ── Rule 5: credit_limit > 0 ──
r5 = df_acc.filter(F.col("credit_limit") <= 0).count()
results.append(("Rule 5", "credit_limit > 0",         r5))

# ── Rule 6: no null transaction_id ──
r6 = df_txn.filter(F.col("transaction_id").isNull()).count()
results.append(("Rule 6", "transaction_id not null",  r6))

# ── Rule 7: no null customer_id ──
r7 = df_txn.filter(F.col("customer_id").isNull()).count()
results.append(("Rule 7", "customer_id not null",     r7))

# ── Rule 8: valid transaction_type ──
valid_types = ["Disbursement","Repayment","Fee","Interest"]
r8 = df_txn.filter(~F.col("transaction_type").isin(valid_types)).count()
results.append(("Rule 8", "valid transaction_type",   r8))

# ── Print Results ──
passed = 0
failed = 0
for rule_id, rule_desc, violations in results:
    status = "✅ PASS" if violations == 0 else "❌ FAIL"
    if violations == 0:
        passed += 1
    else:
        failed += 1
    print(f"  {status} | {rule_id} | {rule_desc:<40} | Violations: {violations:,}")

print("=" * 60)
print(f"  ✅ PASSED : {passed}")
print(f"  ❌ FAILED : {failed}")
print("=" * 60)

  BUSINESS RULE VALIDATION REPORT
  ✅ PASS | Rule 1 | amount >= 0                              | Violations: 0
  ✅ PASS | Rule 2 | outstanding_balance >= 0                 | Violations: 0
  ❌ FAIL | Rule 3 | credit_limit >= outstanding_balance      | Violations: 162,303
  ✅ PASS | Rule 4 | days_past_due >= 0                       | Violations: 0
  ✅ PASS | Rule 5 | credit_limit > 0                         | Violations: 0
  ✅ PASS | Rule 6 | transaction_id not null                  | Violations: 0
  ✅ PASS | Rule 7 | customer_id not null                     | Violations: 0
  ✅ PASS | Rule 8 | valid transaction_type                   | Violations: 0
  ✅ PASSED : 7
  ❌ FAILED : 1


In [0]:
print("=" * 60)
print("  ROW COUNT RECONCILIATION")
print("=" * 60)

bronze_txn  = spark.read.format("delta").load(
    f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions").count()
silver_txn  = df_txn.count()
gold_txn    = fact_txn.count()

print(f"  Bronze transactions  : {bronze_txn:>10,}")
print(f"  Silver transactions  : {silver_txn:>10,}  (removed {bronze_txn - silver_txn:,} dupes)")
print(f"  Gold fact_txn        : {gold_txn:>10,}  (should match silver)")

if silver_txn == gold_txn:
    print("\n  ✅ Silver → Gold row count MATCHES!")
else:
    print(f"\n  ⚠️  Mismatch! Difference = {abs(silver_txn - gold_txn):,} rows")

print("=" * 60)

  ROW COUNT RECONCILIATION
  Bronze transactions  :    986,864
  Silver transactions  :    974,303  (removed 12,561 dupes)
  Gold fact_txn        :    974,303  (should match silver)

  ✅ Silver → Gold row count MATCHES!


In [0]:
# Capture all violations into a reject table for audit
df_rejects = df_txn.filter(
    (F.col("amount") < 0) |
    (F.col("outstanding_balance") < 0) |
    (F.col("transaction_id").isNull()) |
    (F.col("customer_id").isNull()) |
    (~F.col("transaction_type").isin(["Disbursement","Repayment","Fee","Interest"]))
).withColumn("reject_reason",
    F.when(F.col("amount") < 0,                F.lit("negative_amount"))
     .when(F.col("outstanding_balance") < 0,   F.lit("negative_balance"))
     .when(F.col("transaction_id").isNull(),    F.lit("null_transaction_id"))
     .when(F.col("customer_id").isNull(),       F.lit("null_customer_id"))
     .otherwise(F.lit("invalid_transaction_type"))
).withColumn("rejected_at", F.current_timestamp())

df_rejects.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", "true")\
    .save(f"{GOLD}/reject_table")

print(f"✅ Reject table written : {df_rejects.count():,} records")
df_rejects.groupBy("reject_reason").count().show()

✅ Reject table written : 0 records
+-------------+-----+
|reject_reason|count|
+-------------+-----+
+-------------+-----+



In [0]:
from datetime import datetime

audit_log = spark.createDataFrame([
    ("bronze", "transactions",          bronze_txn, "success", str(datetime.now())),
    ("silver", "transactions",          silver_txn, "success", str(datetime.now())),
    ("gold",   "fact_transactions",     gold_txn,   "success", str(datetime.now())),
    ("gold",   "fact_delinquency",
        spark.read.format("delta")
             .load(f"{GOLD}/fact_delinquency").count(), "success", str(datetime.now())),
    ("gold",   "dim_customer",
        spark.read.format("delta")
             .load(f"{GOLD}/dim_customer").count(),     "success", str(datetime.now())),
    ("gold",   "dim_account",
        spark.read.format("delta")
             .load(f"{GOLD}/dim_account").count(),      "success", str(datetime.now())),
],["layer","table_name","row_count","status","processed_at"])

audit_log.write.format("delta").mode("append")\
    .save(f"{GOLD}/audit_log")

print("✅ Audit log written!")
audit_log.show(truncate=False)

✅ Audit log written!
+------+-----------------+---------+-------+--------------------------+
|layer |table_name       |row_count|status |processed_at              |
+------+-----------------+---------+-------+--------------------------+
|bronze|transactions     |986864   |success|2026-03-31 15:39:30.251812|
|silver|transactions     |974303   |success|2026-03-31 15:39:30.251827|
|gold  |fact_transactions|974303   |success|2026-03-31 15:39:30.251832|
|gold  |fact_delinquency |200000   |success|2026-03-31 15:39:30.717717|
|gold  |dim_customer     |100000   |success|2026-03-31 15:39:31.179870|
|gold  |dim_account      |150000   |success|2026-03-31 15:39:31.575864|
+------+-----------------+---------+-------+--------------------------+



In [0]:
print("=" * 60)
print("  VALIDATION COMPLETE — PIPELINE READY!")
print("=" * 60)
print(f"  ✅ Business Rules Passed : {passed}")
print(f"  ❌ Business Rules Failed : {failed}")
print(f"  📋 Reject Records        : {df_rejects.count():,}")
print(f"  📋 Audit Log Entries     : {audit_log.count()}")
print("=" * 60)
print("  🚀 READY FOR AIRFLOW DAG + ADF PIPELINE!")
print("=" * 60)

  VALIDATION COMPLETE — PIPELINE READY!
  ✅ Business Rules Passed : 7
  ❌ Business Rules Failed : 1
  📋 Reject Records        : 0
  📋 Audit Log Entries     : 6
  🚀 READY FOR AIRFLOW DAG + ADF PIPELINE!


In [0]:
print("=" * 60)
print("  WHICH RULE FAILED?")
print("=" * 60)
for rule_id, rule_desc, violations in results:
    if violations > 0:
        print(f"  ❌ {rule_id} | {rule_desc} | Violations: {violations:,}")

# Show sample violating records
print("\n  Sample records violating Rule 3:")
fact_txn.filter(F.col("outstanding_balance") > F.col("credit_limit"))\
    .select("transaction_id","account_id","outstanding_balance","credit_limit")\
    .show(10)

  WHICH RULE FAILED?
  ❌ Rule 3 | credit_limit >= outstanding_balance | Violations: 162,303

  Sample records violating Rule 3:
+--------------+----------+-------------------+------------+
|transaction_id|account_id|outstanding_balance|credit_limit|
+--------------+----------+-------------------+------------+
|       9000020|    534353|          417768.34|      352915|
|       9000455|    566716|          157478.78|      141426|
|       9000682|    631311|         1054007.84|      888313|
|       9000852|    584923|         1089001.37|      983770|
|       9001014|    530598|          314007.81|      266693|
|       9001170|    645561|          267460.37|      236235|
|       9001488|    600853|           773713.0|      651466|
|       9001619|    617546|         1083228.32|      957490|
|       9002058|    642296|          416198.02|      384040|
|       9002194|    642118|          863801.17|      734572|
+--------------+----------+-------------------+------------+
only showing top 1